In [1]:
"""
This notebook cleans the Airbnb listing data. This cleaning phase is all about removing unnecessary columns and prepare the 
data for EDA.

"""

'\nThis notebook cleans the Airbnb listing data. This cleaning phase is all about removing unnecessary columns and prepare the \ndata for EDA.\n\n'

### Load libraries

In [6]:
#Load libraries
import pandas as pd

### Load dataset

In [7]:
# Load dataset

#Load dataset
data_file = "../data/raw/AB_NYC_2019/AB_NYC_2019.csv"
raw_df = pd.read_csv(data_file)

### Remove or keep columns with justification

In [8]:
# identifiers or unstructured text not used as model features. 
# These variables cannot be used as predictive features in a meaningful way.
drop_columns = ['id','name','host_name','host_id'] 

clean_df = raw_df.drop(columns=drop_columns)


The columns name and host_name contain text information that could potentially be used for feature extraction using NLP techniques. However, this project focuses on structured features, so these columns are removed.

### Fix data types

In [31]:
clean_df['last_review'] = pd.to_datetime(clean_df['last_review'], errors="coerce")

### Check data consistency and suspicious values

In [12]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   neighbourhood_group             48895 non-null  object        
 1   neighbourhood                   48895 non-null  object        
 2   latitude                        48895 non-null  float64       
 3   longitude                       48895 non-null  float64       
 4   room_type                       48895 non-null  object        
 5   price                           48895 non-null  int64         
 6   minimum_nights                  48895 non-null  int64         
 7   number_of_reviews               48895 non-null  int64         
 8   last_review                     38843 non-null  datetime64[ns]
 9   reviews_per_month               38843 non-null  float64       
 10  calculated_host_listings_count  48895 non-null  int64         
 11  av

In [19]:
# check impossible values 

numeric_checks = {
    'price_leq_0':(clean_df['price'] < 0).sum(),
    "minimum_nights_less_equal_0": (clean_df["minimum_nights"] <= 0).sum(),
    "number_of_reviews_less_0": (clean_df["number_of_reviews"] < 0).sum(),
    "reviews_per_month_less_0": (clean_df["reviews_per_month"] < 0).sum(),
    "calculated_host_listings_count_less_equal_0": (clean_df["calculated_host_listings_count"] <= 0).sum(),
    "availability_365_outside_range": ((clean_df["availability_365"] < 0) | (clean_df["availability_365"] > 365)).sum()
}

numeric_checks

{'price_leq_0': np.int64(0),
 'minimum_nights_less_equal_0': np.int64(0),
 'number_of_reviews_less_0': np.int64(0),
 'reviews_per_month_less_0': np.int64(0),
 'calculated_host_listings_count_less_equal_0': np.int64(0),
 'availability_365_outside_range': np.int64(0)}

No impossible values. Good!

In [20]:
# suspiciouse values, if any of the below cases are = 0, 
# we should be more careful about how to treat them
zero_checks = {
    "price_eq_0": (clean_df["price"] == 0).sum(),
    "minimum_nights_eq_0": (clean_df["minimum_nights"] == 0).sum(),
    "number_of_reviews_eq_0": (clean_df["number_of_reviews"] == 0).sum(),
    "availability_365_eq_0": (clean_df["availability_365"] == 0).sum()
}

zero_checks

{'price_eq_0': np.int64(11),
 'minimum_nights_eq_0': np.int64(0),
 'number_of_reviews_eq_0': np.int64(10052),
 'availability_365_eq_0': np.int64(17533)}

Having price = 0 and availability_356 = 0 is suspiciouse! 
number_of_reviews = 0 being 10052 is equal to how many NaN values exist in reviews_per_month and last review and explaining that those values are Nan because there is no reviews for them yet
minimum_nights = 0 is realistic

In [47]:
# to check if there are rows with number_of_reviews = 0 but some values 
# for reviews_per_month we check: 
inconsistent_reviews = clean_df[(clean_df["number_of_reviews"] == 0) 
                                & (clean_df["reviews_per_month"].fillna(0) > 0)]

inconsistent_reviews.shape[0]

0

In [46]:
# or if there are rows with number_of_reviews = 0 but some values
# for last_reviews are not NaN
clean_df[(clean_df["number_of_reviews"] == 0) & (clean_df["last_review"].notna())].shape[0]

0

In [24]:
# Any unusual large values?
clean_df[["price","minimum_nights","number_of_reviews","reviews_per_month",
          "calculated_host_listings_count","availability_365"]].describe()

,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


In [33]:
clean_df['neighbourhood_group'].unique()

array(['Brooklyn', 'Manhattan', 'Queens', 'Staten Island', 'Bronx'],
      dtype=object)

In [34]:
clean_df['neighbourhood'].unique()

array(['Kensington', 'Midtown', 'Harlem', 'Clinton Hill', 'East Harlem',
       'Murray Hill', 'Bedford-Stuyvesant', "Hell's Kitchen",
       'Upper West Side', 'Chinatown', 'South Slope', 'West Village',
       'Williamsburg', 'Fort Greene', 'Chelsea', 'Crown Heights',
       'Park Slope', 'Windsor Terrace', 'Inwood', 'East Village',
       'Greenpoint', 'Bushwick', 'Flatbush', 'Lower East Side',
       'Prospect-Lefferts Gardens', 'Long Island City', 'Kips Bay',
       'SoHo', 'Upper East Side', 'Prospect Heights',
       'Washington Heights', 'Woodside', 'Brooklyn Heights',
       'Carroll Gardens', 'Gowanus', 'Flatlands', 'Cobble Hill',
       'Flushing', 'Boerum Hill', 'Sunnyside', 'DUMBO', 'St. George',
       'Highbridge', 'Financial District', 'Ridgewood',
       'Morningside Heights', 'Jamaica', 'Middle Village', 'NoHo',
       'Ditmars Steinway', 'Flatiron District', 'Roosevelt Island',
       'Greenwich Village', 'Little Italy', 'East Flatbush',
       'Tompkinsville', 'Asto

In [35]:
clean_df['room_type'].unique()

array(['Private room', 'Entire home/apt', 'Shared room'], dtype=object)

In [37]:
clean_df["neighbourhood_group"].value_counts(dropna=False)

neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64

In [38]:
clean_df["neighbourhood"].value_counts(dropna=False)

neighbourhood
Williamsburg          3920
Bedford-Stuyvesant    3714
Harlem                2658
Bushwick              2465
Upper West Side       1971
                      ... 
Fort Wadsworth           1
Richmondtown             1
New Dorp                 1
Rossville                1
Willowbrook              1
Name: count, Length: 221, dtype: int64

In [39]:
clean_df["room_type"].value_counts(dropna=False)

room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64

In [42]:
clean_df[["latitude", "longitude"]].describe()

,latitude,longitude
count,48895.000000,48895.000000
mean,40.728949,-73.952170
std,0.054530,0.046157
min,40.499790,-74.244420
25%,40.690100,-73.983070
50%,40.723070,-73.955680
75%,40.763115,-73.936275
max,40.913060,-73.712990


### Handle missing values

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [48]:
clean_df[(clean_df["number_of_reviews"] == 0) 
                    & (clean_df["reviews_per_month"].fillna(0) == 0)].shape[0]


10052

In [49]:
# or if there are rows with number_of_reviews = 0 but some values
# for last_reviews are not NaN
clean_df[(clean_df["number_of_reviews"] == 0) & (clean_df["last_review"].isna())].shape[0]

10052

for futur report:

The variable number_of_reviews was excluded from the feature set because it is strongly related to the target variable (reviews_per_month) and would not be available when predicting popularity for new listings. Including it would lead the model to rely on existing review counts rather than listing characteristics.

In [ ]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from src.config import TARGET_COLUMN

numeric_features = ['price','minimum_nights','calculated_host_listings_count', 'availability_365']
categorical_features = ['neighbourhood_group','room_type']
drop_features = ['number_of_reviews','neighbourhood']


def fill_na_in_target(
    df: pd.DataFrame,
    target = TARGET_COLUMN
) -> pd.DataFrame:
    
    df = df.copy()
    df[target] = df[target].fillna(0)

    return df
    

def build_preprocessing_pipeline() -> object:
    numeric_transformer = StandardScaler()
    categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

    ColumnTransformer(
    transformers=[
        ("num",numeric_transformer, numeric_features),
        ("cat",categorical_transformer, categorical_features),
        ("coords", "passthrough", ["latitude", "longitude"]),
        ("drop_col","drop", drop_features)]
    )

ModuleNotFoundError: No module named 'src'